In [1]:
import os
import json
import io
import base64
from qiskit import qpy
from qc_interactive_education_package import QuantumViewer

# ==========================================
# 1. EXTRACT ENVIRONMENT VARIABLES
# ==========================================
env_mode = os.environ.get("VIEWER_MODE", "sandbox")
env_qubits = os.environ.get("VIEWER_QUBITS")
env_initial = os.environ.get("VIEWER_INITIAL")
env_target = os.environ.get("VIEWER_TARGET")
env_show = os.environ.get("VIEWER_SHOW_CIRCUIT", "1")
env_qpy = os.environ.get("VIEWER_PRELOADED_QPY")
env_gates = os.environ.get("VIEWER_AVAILABLE_GATES")
env_max_gates = os.environ.get("VIEWER_MAX_GATES")

# ==========================================
# 2. PARSE INTO NATIVE OBJECTS
# ==========================================
num_qubits = int(env_qubits) if env_qubits else None
show_circuit = (env_show == "1")

# Handle JSON nulls gracefully
initial_state = None
if env_initial and env_initial != "null":
    initial_state = json.loads(env_initial)

target_state = None
if env_target and env_target != "null":
    target_state = json.loads(env_target)


available_gates = None
if env_gates and env_gates != "null":
    available_gates = json.loads(env_gates)

max_gate_count = int(env_max_gates) if env_max_gates and env_max_gates != "null" else None

# ==========================================
# 3. NATIVE QPY DESERIALIZATION
# ==========================================
# Perfectly resurrects the circuit, custom gate labels, and metadata without decomposition
preloaded_circuit = None
if env_qpy:
    try:
        buf = io.BytesIO(base64.b64decode(env_qpy))
        # qpy.load returns a list of circuits; we take the first one
        preloaded_circuit = qpy.load(buf)[0]
    except Exception as e:
        print(f"Error reconstructing preloaded circuit from QPY: {e}")

# ==========================================
# 4. INSTANTIATE AND RENDER
# ==========================================
app = QuantumViewer(
    mode=env_mode,
    num_qubits=num_qubits,
    initial_state=initial_state,
    target_state=target_state,
    show_circuit=show_circuit,
    preloaded_circuit=preloaded_circuit,
    available_gates=available_gates,
    max_gate_count=max_gate_count
)

app.display()